## Export Qwen3-0.6B to ONNX

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
ONNX_PATH = "qwen3_0.6b.onnx"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

onnx_dir = os.path.dirname(ONNX_PATH)
if onnx_dir and not os.path.exists(onnx_dir):
    os.makedirs(onnx_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.config.use_cache = False 
model.eval().to(DEVICE)

class Qwen3Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, position_ids):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
        )
        return outputs
wrapper = Qwen3Wrapper(model).to(DEVICE)

# export onnx model 
prompt = "Give me a short introduction to large language model."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")

input_ids = model_inputs["input_ids"].to(DEVICE)
attention_mask = model_inputs["attention_mask"].to(DEVICE)
position_ids = torch.clamp(torch.cumsum(attention_mask, dim=1) - 1, min=0)

torch.onnx.export(
    wrapper,
    (input_ids, attention_mask, position_ids),
    ONNX_PATH,
    opset_version=18,
    input_names=["input_ids", "attention_mask", "position_ids"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "position_ids": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"},
    },
)

print(f"ONNX export completed: {ONNX_PATH}")

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 262.05it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/var/folders/mg/3sg4q73x05zbm1lyzkq1_59h0000gn/T/ipykernel_8805/2670963017.py:44: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/var/folders/mg/3sg4q73x05zbm1lyzkq1_59h0000gn/T/ipykernel_8805/2670963017.py:44: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
 

[torch.onnx] Obtain model graph for `Qwen3Wrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Qwen3Wrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/Users/osehn/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


/Users/osehn/Desktop/llm-onnxruntime/.venv/lib/python3.12/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


Applied 308 of general pattern rewrite rules.
ONNX export completed: qwen3_0.6b.onnx


## Validation 

In [6]:
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

onnx_model_path = "qwen3_0.6b.onnx"
hf_model_path = "Qwen/Qwen3-0.6B"
prompt = "Give me a short introduction to large language model."

def run_onnx(model_path, prompt):

    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )

    model_inputs = tokenizer([text], return_tensors="pt")
    input_ids = model_inputs.input_ids.numpy()
    attention_mask = model_inputs.attention_mask.numpy()

    position_ids = np.cumsum(attention_mask, axis=1) - 1
    position_ids = np.clip(position_ids, 0, None)

    feeds = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "position_ids": position_ids,
    }

    outs = sess.run(None, feeds)
    logits_onnx = torch.from_numpy(outs[0])

    return logits_onnx

def run_hf(model_path, prompt):
    model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )

    model_inputs = tokenizer([text], return_tensors="pt")

    with torch.no_grad(): 
        outputs = model(**model_inputs)

    return outputs.logits

if __name__ == "__main__":

    logits_onnx = run_onnx(onnx_model_path, prompt)
    logits_hf = run_hf(hf_model_path, prompt)

    sqnr_last_logits = 20 * torch.log10(
        torch.norm(logits_hf[:, -1, :]) / torch.norm(logits_hf[:, -1, :] - logits_onnx[:, -1, :])
    )
    print(f"SQNR(last logits): {sqnr_last_logits.item():.2f} dB")

Loading weights: 100%|██████████| 311/311 [00:02<00:00, 138.17it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


SQNR(last logits): 112.30 dB


### LLM with onnxruntime 

In [8]:
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

def run_onnx_logits(sess, input_ids, attention_mask):
    # input_ids, attention_mask: numpy int64, shape [B, T]
    position_ids = np.cumsum(attention_mask, axis=1) - 1
    position_ids = np.clip(position_ids, 0, None).astype(np.int64)

    feeds = {
        "input_ids": input_ids.astype(np.int64),
        "attention_mask": attention_mask.astype(np.int64),
        "position_ids": position_ids,
    }

    (logits,) = sess.run(None, feeds)  # [B, T, V]
    return logits

def generate_onnx(model_path, prompt, max_new_tokens=50):
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
    eos_token_id = tokenizer.eos_token_id

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="np")
    input_ids = model_inputs["input_ids"].astype(np.int64)
    attention_mask = model_inputs["attention_mask"].astype(np.int64)

    prompt_len = input_ids.shape[1]

    for _ in range(max_new_tokens):
        logits = run_onnx_logits(sess, input_ids, attention_mask)
        next_token = np.argmax(logits[:, -1, :], axis=-1).astype(np.int64)  # [B]

        # append
        input_ids = np.concatenate([input_ids, next_token[:, None]], axis=1)
        attention_mask = np.concatenate(
            [attention_mask, np.ones((attention_mask.shape[0], 1), dtype=np.int64)],
            axis=1,
        )

        if eos_token_id is not None and next_token[0] == eos_token_id:
            break

    generated_only = tokenizer.decode(input_ids[0, prompt_len:], skip_special_tokens=True)
    full_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return generated_only, full_text

gen, full = generate_onnx("qwen3_0.6b.onnx", "Give me a short introduction to large language model.", 80)
print(gen)



Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know about LLMs. They're big language models, right? So I should mention their ability to understand and generate text. Maybe start with the basics: they're trained on massive datasets to understand a lot of information.

I need to highlight their key features. Like,


### LLM with huggingface

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

/Users/osehn/Desktop/llm-onnxruntime/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 271.47it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


thinking content: Okay, the user wants a short introduction to a large language model. Let me start by understanding what they need. They might be a student, a researcher, or someone learning about AI. They probably need a concise yet informative overview.

First, I should mention that LLMs are artificial intelligence systems. Then, highlight their ability to understand and generate text. Maybe include examples like answering questions or writing articles. It's important to mention the different types of LLMs, like transformer models. Also, touch on their applications in various fields. Keep it simple and avoid technical jargon. Make sure to end with a positive note about their potential impact. Let me check if I covered all key points without being too long. Yeah, that should work.
</think>
content: A large language model (LLM) is a type of artificial intelligence designed to understand and generate human language. These models, often based on deep learning techniques, can process tex